# 12 - External Evaluation Plan for L78 Hybrid

Prepara avaliação externa com `lm-evaluation-harness`. Este notebook instala dependências, documenta comandos e salva um plano executável. A execução completa pode ser pesada no Colab.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, math

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    !pip -q install transformers accelerate sentencepiece safetensors peft datasets
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
UTILS_DIR = PROJECT_ROOT / 'utils'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, UTILS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)

OUT_DIR = RESULTS_DIR / 'external_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)


In [ ]:
# Instalação opcional. Rode se for executar benchmarks externos nesta sessão.
!pip -q install lm-eval accelerate sentencepiece peft transformers safetensors


In [ ]:
# Configuração recomendada inicial.
# Começar pequeno para validar o pipeline antes de rodar MMLU completo.
MODEL_ARGS = f'pretrained={CHECKPOINTS_DIR / "student_ternary_recovered_l78"}'
TASKS_FAST = ['arc_easy', 'hellaswag']
TASKS_FULL = ['arc_easy', 'arc_challenge', 'hellaswag', 'truthfulqa_mc1']
BATCH_SIZE = 1
LIMIT = 100  # use None para full benchmark

commands = []
for tasks, label in [(TASKS_FAST, 'fast'), (TASKS_FULL, 'full')]:
    task_str = ','.join(tasks)
    limit_part = f' --limit {LIMIT}' if LIMIT else ''
    out_file = OUT_DIR / f'l78_hybrid_{label}.json'
    cmd = f'lm_eval --model hf --model_args "{MODEL_ARGS}" --tasks {task_str} --batch_size {BATCH_SIZE}{limit_part} --output_path {out_file}'
    commands.append({'label': label, 'tasks': tasks, 'command': cmd, 'output_path': str(out_file)})

print(json.dumps(commands, ensure_ascii=False, indent=2))


In [ ]:
plan = {
    'model': 'student_ternary_recovered_l78 adapter checkpoint',
    'model_args': MODEL_ARGS,
    'commands': commands,
    'interpretation': {
        'purpose': 'Check whether local 0.92 recovery generalizes beyond the 100-question suite.',
        'compare_against': ['base_reference', 'student_pruned', 'pruned_8bit', 'l78_hybrid'],
        'warning': 'This does not benchmark packed ternary runtime speed; it benchmarks model quality.'
    }
}
plan_path = OUT_DIR / 'external_eval_plan.json'
plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', plan_path)


## Execução

Primeiro rode o comando `fast`. Se funcionar, rode `full`. Depois envie os JSONs gerados em `results/external_eval/`.